### Deep symbolic regression
Use python 3.7 to run the code since it doesn't work with newer versions

In [ ]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

from dso import DeepSymbolicOptimizer



In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('../data/raw/complete_data/complete_samples_up_until_020625.csv')

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.sampling.sampler import PARAMETER_RANGES

In [ ]:
ucell_cols = [f'Ucell_{i}' for i in range(31)]
ifc_cols = [f'ifc_{i}' for i in range(31)]

# Combine into long format
df_long = df.melt(
    id_vars=[col for col in df.columns if col not in ucell_cols + ifc_cols],
    value_vars=ucell_cols,
    var_name='step',
    value_name='Ucell'
)

df_long['ifc'] = df.melt(
    id_vars=[col for col in df.columns if col not in ucell_cols + ifc_cols],
    value_vars=ifc_cols,
    value_name='ifc'
)['ifc']

In [ ]:
data = df_long[df_long['step'] == "Ucell_0"]
data

In [ ]:
cols = list(PARAMETER_RANGES.keys())

X = data[cols].dropna()
y = data['Ucell'].dropna()

X_np = X.to_numpy()
y_np = y.to_numpy()

In [ ]:
from sklearn.model_selection import train_test_split

# Assuming X is a 2D NumPy array and y is a 1D NumPy array
X_train, X_test, y_train, y_test = train_test_split(
    X_np, y_np, test_size=0.2, random_state=42
)

from dso import DeepSymbolicRegressor
model = DeepSymbolicRegressor(config='DSR_conf.json',)
model.fit(X_train, y_train)


In [ ]:
# Prediction and evaluation
y_pred = model.predict(X_test)

In [ ]:
# After fitting the model
print(model.program_.pretty())


In the following we fit a quadratic regression model to the data (Ucell = a^2ifc + b ifc + c) and then learn formulas for a, b and c using DSR.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

# Define parametric family
def quadratic(t, a, b, c):
    return a * t**2 + b * t + c

# Identify columns
ifc_cols = [f"ifc_{i}" for i in range(31)]
ucell_cols = [f"Ucell_{i}" for i in range(31)]
input_cols = list(PARAMETER_RANGES.keys())

# Prepare result storage
params_list = []
inputs_list = []

for idx, row in df.iterrows():
    t = row[ifc_cols].values.astype(float)
    y = row[ucell_cols].values.astype(float)
    inputs = row[input_cols]

    try:
        popt, _ = curve_fit(quadratic, t, y, maxfev=10000)
        params_list.append(popt)  # [a, b, c]
        inputs_list.append(inputs.values)
    except RuntimeError:
        params_list.append([np.nan, np.nan, np.nan])
        inputs_list.append(inputs.values)

# Create DataFrames
df_inputs = pd.DataFrame(inputs_list, columns=input_cols)
df_params = pd.DataFrame(params_list, columns=["a", "b", "c"])
df_model = pd.concat([df_inputs, df_params], axis=1)


In [ ]:
from pysr import PySRRegressor

X = df_inputs.values  # shape (n_samples, n_features)

# Target vectors
y_a = df_params['a'].values
y_b = df_params['b'].values
y_c = df_params['c'].values

# Define symbolic regression models
sr_a = PySRRegressor(niterations=1000, model_selection="best")
sr_b = PySRRegressor(niterations=1000, model_selection="best")
sr_c = PySRRegressor(niterations=1000, model_selection="best")

# Fit the models
sr_a.fit(X, y_a)
sr_b.fit(X, y_b)
sr_c.fit(X, y_c)


In [ ]:
print("a(x) =", sr_a.get_best())
print("b(x) =", sr_b.get_best())
print("c(x) =", sr_c.get_best())


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_sr_vs_true(df, df_inputs, sr_a, sr_b, sr_c, ucell_cols, ifc_cols, sample_indices=None):
    if sample_indices is None:
        sample_indices = range(5)  # default: first 5

    for idx in sample_indices:
        # Original data
        t = df.loc[idx, ifc_cols].values.astype(float)
        y_true = df.loc[idx, ucell_cols].values.astype(float)

        # SR prediction of coefficients
        x_input = df_inputs.iloc[idx].values.reshape(1, -1)
        a = sr_a.predict(x_input)[0]
        b = sr_b.predict(x_input)[0]
        c = sr_c.predict(x_input)[0]

        # Predicted curve
        t_fit = np.linspace(t.min(), t.max(), 200)
        y_pred = a * t_fit**2 + b * t_fit + c

        # Plot
        plt.figure(figsize=(6, 4))
        plt.plot(t, y_true, 'o', label='Original', markersize=4)
        plt.plot(t_fit, y_pred, '-', label='SR prediction', linewidth=2)
        plt.title(f"Sample {idx}")
        plt.xlabel("ifc")
        plt.ylabel("Ucell")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

plot_sr_vs_true(
    df=df,
    df_inputs=df_inputs,
    sr_a=sr_a,
    sr_b=sr_b,
    sr_c=sr_c,
    ucell_cols=ucell_cols,
    ifc_cols=ifc_cols,
    sample_indices=range(len(df))  # or choose worst-fit examples
)


In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

errors = []

for idx in range(len(df)):
    t = df.loc[idx, ifc_cols].values.astype(float)
    y_true = df.loc[idx, ucell_cols].values.astype(float)
    x_input = df_inputs.iloc[idx].values.reshape(1, -1)

    a = sr_a.predict(x_input)[0]
    b = sr_b.predict(x_input)[0]
    c = sr_c.predict(x_input)[0]

    y_pred = a * t**2 + b * t + c
    mse = mean_squared_error(y_true, y_pred)
    errors.append((idx, mse))

df_errors = pd.DataFrame(errors, columns=["index", "mse"])


In [ ]:
best_10_indices = df_errors.sort_values("mse", ascending=True).head(10)["index"].tolist()
import matplotlib.pyplot as plt

def plot_sr_vs_true_best_10(df, df_inputs, sr_a, sr_b, sr_c, ucell_cols, ifc_cols, sample_indices):
    for idx in sample_indices:
        t = df.loc[idx, ifc_cols].values.astype(float)
        y_true = df.loc[idx, ucell_cols].values.astype(float)
        x_input = df_inputs.iloc[idx].values.reshape(1, -1)

        a = sr_a.predict(x_input)[0]
        b = sr_b.predict(x_input)[0]
        c = sr_c.predict(x_input)[0]

        t_fit = np.linspace(t.min(), t.max(), 200)
        y_fit = a * t_fit**2 + b * t_fit + c

        plt.figure(figsize=(6, 4))
        plt.plot(t, y_true, 'o', label="True", markersize=4)
        plt.plot(t_fit, y_fit, '-', label="SR prediction", linewidth=2)
        plt.title(f"Best Fit Sample {idx} - MSE: {df_errors.loc[df_errors['index'] == idx, 'mse'].values[0]:.4f}")
        plt.xlabel("ifc")
        plt.ylabel("Ucell")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

plot_sr_vs_true_best_10(
    df=df,
    df_inputs=df_inputs,
    sr_a=sr_a,
    sr_b=sr_b,
    sr_c=sr_c,
    ucell_cols=ucell_cols,
    ifc_cols=ifc_cols,
    sample_indices=best_10_indices
)
